# 03c – Random Forest Regressor
**Owner:** Member 3 (Classical Machine Learning Engineer)

Tunes a `RandomForestRegressor` inside the shared preprocessing `Pipeline` using `RandomizedSearchCV` on training data only, logs CV & test metrics, and extracts feature importances.

In [1]:
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CV_FOLDS, FIGURES_DIR, MODELS_DIR, RANDOM_SEED, RESULTS_DIR
from src.evaluate import (
    compute_metrics,
    cross_validate_model,
    plot_parity,
    plot_residuals,
    save_model_run_metadata,
    save_predictions,
)
from src.features import SELECTED_FEATURES, build_preprocessor, get_xy

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

X_train, y_train, id_train = get_xy("train")
X_val, y_val, id_val = get_xy("val")
X_test, y_test, id_test = get_xy("test")

In [2]:
rf_pipe = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)),
])

param_dist = {
    "model__n_estimators": [200, 350, 500],
    "model__max_depth": [12, 18, 25, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", 0.3, 0.5, 0.7],
}

cv_split = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=param_dist,
    n_iter=15,
    cv=cv_split,
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_SEED,
    n_jobs=-1,
    refit=False,
    verbose=1,
)

t_search_start = time.perf_counter()
search.fit(X_train, y_train)
search_time = time.perf_counter() - t_search_start

print(f"Search completed in {search_time:.2f} s")
print("Best CV RMSE:", round(-search.best_score_, 4))
print("Best Params :", search.best_params_)

Fitting 5 folds for each of 15 candidates, totalling 75 fits


Search completed in 164.07 s
Best CV RMSE: 0.9942
Best Params : {'model__n_estimators': 500, 'model__min_samples_split': 5, 'model__min_samples_leaf': 1, 'model__max_features': 0.5, 'model__max_depth': 25}


In [3]:
best_rf_params = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
best_rf_pipe = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1, **best_rf_params)),
])

cv_summary = cross_validate_model(best_rf_pipe, X_train, y_train, cv=CV_FOLDS)
print("Final RF 5-Fold CV:", {k: round(v, 4) for k, v in cv_summary.items()})

t_fit_start = time.perf_counter()
best_rf_pipe.fit(X_train, y_train)
fit_time = time.perf_counter() - t_fit_start

val_pred = best_rf_pipe.predict(X_val)
val_metrics = compute_metrics(y_val, val_pred)
print(f"Validation Metrics: {val_metrics}")

test_pred = best_rf_pipe.predict(X_test)
test_metrics = compute_metrics(y_test, test_pred)
print(f"Test Metrics      : {test_metrics}")

joblib.dump(best_rf_pipe, MODELS_DIR / "random_forest_best.joblib")
save_predictions(id_val, y_val, val_pred, "random_forest", "val")
save_predictions(id_test, y_test, test_pred, "random_forest", "test")
save_model_run_metadata(
    model_name="random_forest",
    display_name="Random Forest",
    training_time_sec=fit_time,
    search_time_sec=search_time,
    cv_summary=cv_summary,
    extra={"best_params": search.best_params_},
)

# Extract and save feature importances
importances = best_rf_pipe.named_steps["model"].feature_importances_
fi_df = pd.DataFrame({"feature": SELECTED_FEATURES, "importance": importances})
fi_df = fi_df.sort_values("importance", ascending=False).reset_index(drop=True)
fi_df.to_csv(RESULTS_DIR / "rf_feature_importance.csv", index=False)

fig, ax = plt.subplots(figsize=(8.5, 6))
top20 = fi_df.head(20).iloc[::-1]
ax.barh(top20["feature"], top20["importance"], color="#2a9d8f")
ax.set_title("Random Forest Top 20 Feature Importances")
ax.set_xlabel("Mean Decrease in Impurity (Importance)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "rf_feature_importance.png", dpi=200)
plt.close(fig)

plot_parity(y_test, test_pred, "random_forest", split="test")
plot_residuals(y_test, test_pred, "random_forest", split="test")
print(fi_df.head(10))

Final RF 5-Fold CV: {'r2_mean': 0.5778, 'r2_std': 0.0272, 'rmse_mean': 0.9942, 'rmse_std': 0.04, 'mae_mean': 0.6916, 'mae_std': 0.0225}


Validation Metrics: {'r2': 0.6269475234919473, 'rmse': 1.0252174941239045, 'mae': 0.6990413535695417}


Test Metrics      : {'r2': 0.5835279373722216, 'rmse': 0.9442078975277444, 'mae': 0.6582845576442437}


            feature  importance
0  mc_CRY-chi-2-all    0.082291
1  mc_CRY-chi-0-all    0.076538
2  density [g/cm^3]    0.033722
3                Di    0.032222
4                Df    0.031209
5     POAV [cm^3/g]    0.021323
6         f-lig-Z-1    0.020354
7    mc_CRY-S-1-all    0.018971
8  mc_CRY-chi-1-all    0.018035
9    mc_CRY-Z-0-all    0.017119
